In [ ]:
import geopandas as gpd
import pandana
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TARGET_CRS = "EPSG:32618"  # UTM Zone 18N — meters

# Load
candidates = gpd.read_file(PROJECT_ROOT / "data/processed/candidate_sites_filtered.gpkg")
nodes = gpd.read_file(PROJECT_ROOT / "data/raw/manhattan_nodes.gpkg")
edges = gpd.read_file(PROJECT_ROOT / "data/raw/manhattan_roads.gpkg")

# Reproject everything to meters
candidates = candidates.to_crs(TARGET_CRS)
nodes = nodes.to_crs(TARGET_CRS)
edges = edges.to_crs(TARGET_CRS)

# Verify
print(f"Candidates CRS: {candidates.crs}")
print(f"Nodes CRS: {nodes.crs}")
print(f"Edges CRS: {edges.crs}")
print(f"Sample edge length: {edges.geometry.length.iloc[0]:.2f} meters")

In [ ]:
import pandana
print(pandana.__file__)

In [ ]:
import numpy as np

nodes["node_id"] = np.arange(len(nodes), dtype=np.int32)
osmid_to_nodeid = dict(zip(nodes["osmid"], nodes["node_id"]))
nodes = nodes.set_index("node_id")

nodes["x"] = nodes.geometry.x
nodes["y"] = nodes.geometry.y

edges["length_m"] = edges.geometry.length
edges["from_id"] = edges["u"].map(osmid_to_nodeid)
edges["to_id"] = edges["v"].map(osmid_to_nodeid)

edges = edges.dropna(subset=["from_id", "to_id"])
edges["from_id"] = edges["from_id"].astype(np.int32)
edges["to_id"] = edges["to_id"].astype(np.int32)

network = pandana.Network(
    node_x=nodes["x"],
    node_y=nodes["y"],
    edge_from=edges["from_id"],
    edge_to=edges["to_id"],
    edge_weights=edges[["length_m"]],
)

print(f"Network built: {len(nodes):,} nodes, {len(edges):,} edges")

In [ ]:

# Step 2 — Euclidean pre-filter
import time
import numpy as np
from scipy.spatial import cKDTree

t2_start = time.time()

coords = np.column_stack([candidates.geometry.x.values, candidates.geometry.y.values])
tree = cKDTree(coords)

# Returns all (i, j) pairs with i < j and Euclidean dist <= 700m
pairs = tree.query_pairs(700.0, output_type="ndarray")

t2 = time.time() - t2_start
print(f"Step 2: {len(pairs):,} candidate pairs within 700m Euclidean distance")
print(f"Step 2 runtime: {t2:.2f}s")


In [ ]:

# Step 3 — Network distance computation via pandana nearest_pois
# Any pair within ≤500m network distance has Euclidean distance ≤500m ≤700m,
# so it is guaranteed to be in the pre-filtered set from Step 2.
import pandas as pd

t3_start = time.time()

N_POIS = 200  # safe upper bound for candidates within 500m in Manhattan

# Register candidates as POIs (maxdist=700 matches Euclidean pre-filter)
x_s = pd.Series(candidates.geometry.x.values)   # index 0..n-1 → POI ID = candidate index
y_s = pd.Series(candidates.geometry.y.values)
network.set_pois("cand", maxdist=700, maxitems=N_POIS, x_col=x_s, y_col=y_s)

# nearest_pois with include_poi_ids=True returns ONE wide DataFrame:
#   distance columns  : 1, 2, ..., N_POIS
#   POI index columns : poi1, poi2, ..., poi{N_POIS}
combined = network.nearest_pois(500, "cand", num_pois=N_POIS, include_poi_ids=True)

dist_cols = list(range(1, N_POIS + 1))
id_cols   = [f"poi{k}" for k in range(1, N_POIS + 1)]

# Snap candidates to nearest network nodes.
# No mapping_distance cutoff: get_node_ids drops rows beyond the cutoff,
# which would shorten the Series and break positional indexing.
raw_nodes = network.get_node_ids(candidates.geometry.x, candidates.geometry.y)
# Reindex to candidates.index so we get exactly one entry per candidate (fill=-1 for any gaps)
node_arr = raw_nodes.reindex(candidates.index, fill_value=-1).astype(int).values

# Build coverage dict: i → [j, ...] (j covers i within 500m network distance)
coverage = {}
for i in range(len(candidates)):
    nd = node_arr[i]
    if nd < 0 or nd not in combined.index:
        coverage[i] = [i]
        continue
    row = combined.loc[nd]
    dists = row[dist_cols]
    ids   = row[id_cols]
    valid = dists.notna() & (dists <= 500)
    js = ids[valid.values].dropna().astype(int).tolist()
    coverage[i] = js if js else [i]

t3 = time.time() - t3_start
sizes = [len(v) for v in coverage.values()]
print(f"Coverage matrix: {len(coverage):,} demand points")
print(f"  Mean covering sites per demand point: {np.mean(sizes):.1f}")
print(f"  Max: {max(sizes)}, Min: {min(sizes)}")
print(f"Step 3 runtime: {t3:.2f}s")


In [ ]:

# Step 4 — MCLP optimisation with PuLP
import pulp

def run_mclp(candidates_gdf, coverage_matrix, p):
    t0 = time.time()
    n = len(candidates_gdf)
    nkde = candidates_gdf["nkde_score"].values

    prob = pulp.LpProblem(f"MCLP_p{p}", pulp.LpMaximize)

    x = pulp.LpVariable.dicts("x", range(n), cat="Binary")
    # y_i can be continuous [0,1]: with non-negative weights and the coverage
    # constraint sum(x_j) >= y_i, the LP always drives y_i to 0 or 1 exactly,
    # so no integrality is lost but CBC has half as many binary vars to branch on.
    y = pulp.LpVariable.dicts("y", range(n), lowBound=0, upBound=1, cat="Continuous")

    # Objective: maximise weighted demand covered
    prob += pulp.lpSum(nkde[i] * y[i] for i in range(n))

    # Coverage: y_i can only reach 1 if at least one covering facility is selected
    for i, js in coverage_matrix.items():
        prob += pulp.lpSum(x[j] for j in js) >= y[i]

    # Facility count
    prob += pulp.lpSum(x[j] for j in range(n)) == p

    # gapRel=0.005: stop when the best integer solution is within 0.5% of the LP bound.
    # timeLimit=300: hard ceiling of 5 minutes per p so the loop never hangs.
    prob.solve(pulp.PULP_CBC_CMD(msg=1, gapRel=0.005, timeLimit=300))

    sel_idx = [j for j in range(n) if (pulp.value(x[j]) or 0) > 0.5]
    cov_dem = sum(nkde[i] for i in range(n) if (pulp.value(y[i]) or 0) > 0.5)
    total_dem = nkde.sum()
    pct = cov_dem / total_dem * 100

    sel_gdf = candidates_gdf.iloc[sel_idx].copy().reset_index(drop=True)
    return sel_gdf, cov_dem, pct, time.time() - t0


p_values = [50, 75, 100, 125, 150]
results = {}
total_demand = candidates["nkde_score"].sum()

for p in p_values:
    sel_gdf, cov_dem, pct, runtime = run_mclp(candidates, coverage, p)
    results[p] = {
        "selected_gdf": sel_gdf,
        "covered_demand": cov_dem,
        "pct_covered": pct,
    }
    print(
        f"p={p:3d}: {len(sel_gdf)} sites selected | "
        f"demand covered = {cov_dem:.2f} ({pct:.1f}%) | "
        f"Step 4 runtime = {runtime:.1f}s"
    )


In [ ]:

# Step 5 — Coverage curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

p_vals = sorted(results.keys())
pcts = [results[p]["pct_covered"] for p in p_vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_vals, pcts, "o-", color="steelblue", linewidth=2, markersize=8)
ax.set_xlabel("Number of PUDO Sites (p)", fontsize=12)
ax.set_ylabel("% Weighted Demand Covered", fontsize=12)
ax.set_title("MCLP Coverage Curve — Manhattan AV PUDO Sites", fontsize=13)
ax.set_xticks(p_vals)
ax.grid(True, alpha=0.3)
for p, pct in zip(p_vals, pcts):
    ax.annotate(f"{pct:.1f}%", (p, pct), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=10)
plt.tight_layout()
out_img = PROJECT_ROOT / "data" / "outputs" / "coverage_curve.png"
plt.savefig(out_img, dpi=150)
plt.show()
print(f"Saved: {out_img.name}")


In [ ]:

# Step 6 — Save selected sites for each p
out_dir = PROJECT_ROOT / "data" / "processed"
for p in p_values:
    out_path = out_dir / f"mclp_selected_p{p}.gpkg"
    results[p]["selected_gdf"].to_file(out_path, driver="GPKG")
    print(f"Saved: {out_path.name}  ({len(results[p]['selected_gdf'])} sites)")
